# Phase 1: 데이터 구축 및 전처리

## 목적
- 전국 단위 데이터에서 서울시 데이터만 추출
- 모든 공간 데이터를 EPSG:5179 좌표계로 통일
- 공원 중심점 생성 및 네트워크 분석 준비
- 행정동별 인구 데이터 통합

## 데이터 출처
1. 행정경계: 브이월드 센서스경계 (SHP)
2. 공원: 서울시 도시공원 현황 (CSV)
3. 인구: 서울시 행정동별 인구 (XLSX)
4. NDVI: Landsat 9 위성영상 (TIF)
5. 도로: OSM 보행자 네트워크 (추후 다운로드)

In [ ]:
# 필수 라이브러리 임포트
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

In [ ]:
# 경로 설정# 상대 경로 사용 (현재 파일 기준)base_path = Path(__file__).parent.parent / 's1_data' / 's1_org_data' if '__file__' in dir() else Path('../s1_data/s1_org_data')output_path = Path(__file__).parent.parent / 's1_data' / 's2_processed_data' if '__file__' in dir() else Path('../s1_data/s2_processed_data')output_path.mkdir(parents=True, exist_ok=True)print(f"원본 데이터 경로: {base_path}")print(f"처리된 데이터 저장 경로: {output_path}")

## 1. 행정경계 데이터 구축
### 1-1. 전국 행정동 경계에서 서울시만 추출

In [ ]:
# 시도 경계 로드
sd_path = os.path.join(base_path, '브이월드', '센서스경계', 'BND_SIDO_PG')
sd_files = list(Path(sd_path).glob('*.shp'))
print(f"발견된 Shapefile: {len(sd_files)}개")
for f in sd_files:
    print(f"  - {f.name}")
gdf_sd_all = gpd.read_file(sd_files[0], encoding='cp949')

print(f"\n전체 데이터 행 수: {len(gdf_sd_all)}")
print(f"컬럼: {list(gdf_sd_all.columns)}")
print(f"\n원본 좌표계: {gdf_sd_all.crs}")
gdf_sd_all.head()

In [ ]:
# 시군구 경계 로드
sgg_path = os.path.join(base_path, '브이월드', '센서스경계', 'BND_SIGUNGU_PG')
sgg_files = list(Path(sgg_path).glob('*.shp'))
print(f"발견된 Shapefile: {len(sgg_files)}개")
for f in sgg_files:
    print(f"  - {f.name}")
gdf_sgg_all = gpd.read_file(sgg_files[0], encoding='cp949')

print(f"\n전체 데이터 행 수: {len(gdf_sgg_all)}")
print(f"컬럼: {list(gdf_sgg_all.columns)}")
print(f"\n원본 좌표계: {gdf_sgg_all.crs}")
gdf_sgg_all.head()

In [ ]:
# 행정동 경계 로드
dong_path = os.path.join(base_path, '브이월드', '센서스경계', 'BND_ADM_DONG_PG')
#dong_files = list(dong_path.glob('*.shp'))
dong_files = list(Path(dong_path).glob('*.shp'))
print(f"발견된 Shapefile: {len(dong_files)}개")
for f in dong_files:
    print(f"  - {f.name}")

# 첫 번째 shapefile 로드 (행정동 경계로 추정)
gdf_dong_all = gpd.read_file(dong_files[0], encoding='cp949')

print(f"\n전체 데이터 행 수: {len(gdf_dong_all)}")
print(f"컬럼: {list(gdf_dong_all.columns)}")
print(f"\n원본 좌표계: {gdf_dong_all.crs}")
gdf_dong_all.head()

In [ ]:
# 서울시 데이터만 필터링
# 컬럼명이 다를 수 있으므로 먼저 확인
gdf_seoul_sd = gdf_sd_all[gdf_sd_all['SIDO_CD'].astype(str).str.startswith('11')]
gdf_seoul_sgg = gdf_sgg_all[gdf_sgg_all['SIGUNGU_CD'].astype(str).str.startswith('11')]
gdf_seoul_dong = gdf_dong_all[gdf_dong_all['ADM_CD'].astype(str).str.startswith('11')]


# 행정동 데이터에 시군구명 병합
gdf_seoul_dong["SIGUNGU_CD"] = gdf_seoul_dong["ADM_CD"].str[:5]
gdf_seoul_dong.head()
gdf_seoul_sgg_dong = gdf_seoul_dong.merge(gdf_seoul_sgg[['SIGUNGU_CD', 'SIGUNGU_NM']], on='SIGUNGU_CD', how='left')

# 시도명 추가 
gdf_seoul_sgg_dong['SIDO_NM'] = '서울특별시'
gdf_seoul_sgg_dong['SIDO_CD'] = '11'
gdf_seoul = gdf_seoul_sgg_dong[['SIDO_CD','SIDO_NM','SIGUNGU_CD', 'SIGUNGU_NM','ADM_CD', 'ADM_NM' , 'geometry']]
gdf_seoul.head()

### 1-2. 좌표계를 EPSG:5179로 변환

In [ ]:
# EPSG:5179 (Korea 2000 / Central Belt)로 변환
if gdf_seoul.crs != 'EPSG:5179':
    gdf_seoul = gdf_seoul.to_crs('EPSG:5179')
    print(f"좌표계 변환 완료: {gdf_seoul.crs}")
else:
    print(f"이미 EPSG:5179 좌표계입니다.")

In [ ]:
# 1) 행정동별 면적
gdf_seoul["adm_area_m2"] = gdf_seoul.geometry.area
gdf_seoul["adm_area_km2"] = gdf_seoul["adm_area_m2"] / 1_000_000

adm_area = gdf_seoul[[
    "SIDO_CD","SIDO_NM","SIGUNGU_CD","SIGUNGU_NM","ADM_CD","ADM_NM",
    "adm_area_m2","adm_area_km2"
]].copy()
adm_area.head()

# 2) 시군구별 면적 (행정동을 시군구 경계로 합친 뒤 면적)
sigungu_gdf = gdf_seoul.dissolve(
    by=["SIDO_CD","SIDO_NM","SIGUNGU_CD","SIGUNGU_NM"],
    as_index=False
)
sigungu_gdf["sigungu_area_m2"] = sigungu_gdf.geometry.area
sigungu_gdf["sigungu_area_km2"] = sigungu_gdf["sigungu_area_m2"] / 1_000_000

sigungu_area = sigungu_gdf[[
    "SIDO_CD","SIDO_NM","SIGUNGU_CD","SIGUNGU_NM",
    "sigungu_area_m2","sigungu_area_km2"
]].copy()

# 3) 서울 전체 면적 (행정동을 시도 경계로 합친 뒤 면적)
sido_gdf = gdf_seoul.dissolve(by=["SIDO_CD","SIDO_NM"], as_index=False)
sido_gdf["sido_area_m2"] = sido_gdf.geometry.area
sido_gdf["sido_area_km2"] = sido_gdf["sido_area_m2"] / 1_000_000

seoul_total_km2 = float(sido_gdf.loc[0, "sido_area_km2"])

print("서울 전체 면적(km²):", seoul_total_km2)

In [ ]:
# 시각화: 서울시 행정동 경계
fig, ax = plt.subplots(figsize=(12, 10))
gdf_seoul.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
ax.set_title('서울시 행정동 경계', fontsize=16, fontweight='bold')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"총 행정동 수: {len(gdf_seoul)}개")

## 2. 공원 데이터 구축
### 2-1. 서울시 도시공원 데이터 로드 및 전처리

In [ ]:
# 공원 데이터 로드
park_shp = base_path / '서울공원정보' / 'UPIS_SHP_ZON216.shp'
df_park = gpd.read_file(park_shp)

print(f"공원 데이터 행 수: {len(df_park)}")
print(f"컬럼: {list(df_park.columns)}")
print(f"\n상위 5개 데이터:")
df_park.head()

In [ ]:
df_park["df_park_area_m2"] = df_park.geometry.area
df_park["df_park_area_km2"] = df_park["df_park_area_m2"] / 1_000_000
#df_park.head()

print(df_park[df_park.columns[df_park.columns.str.contains('df_park_area_m2', na=False)]].describe())
# 결측치 확인
print(f"\n결측치 현황:")
print(df_park.isnull().sum())

In [ ]:
# 공원 위치 정보가 있는 컬럼 확인 (위도, 경도 또는 geometry)
# CSV 파일이므로 좌표 컬럼을 찾아서 GeoDataFrame으로 변환 필요

# 위도/경도 컬럼 찾기
coord_cols = [col for col in df_park.columns if any(keyword in col for keyword in ['위도', '경도', 'LAT', 'LON', 'X', 'Y'])]
print(f"좌표 관련 컬럼: {coord_cols}")

if len(coord_cols) >= 2:
    # 위도/경도가 있는 경우
    lat_col = [c for c in coord_cols if '위도' in c or 'LAT' in c.upper()][0]
    lon_col = [c for c in coord_cols if '경도' in c or 'LON' in c.upper()][0]
    
    # 결측치 제거
    df_park_valid = df_park.dropna(subset=[lat_col, lon_col])
    
    # GeoDataFrame 생성 (WGS84 좌표계)
    gdf_park = gpd.GeoDataFrame(
        df_park_valid,
        geometry=gpd.points_from_xy(df_park_valid[lon_col], df_park_valid[lat_col]),
        crs='EPSG:4326'
    )
    
    # EPSG:5179로 변환
    gdf_park = gdf_park.to_crs('EPSG:5179')
    
    print(f"\n공원 GeoDataFrame 생성 완료")
    print(f"유효한 공원 수: {len(gdf_park)}")
    print(f"좌표계: {gdf_park.crs}")
    
else:
    print("경고: 좌표 컬럼을 찾을 수 없습니다. 공원 위치 정보를 수동으로 확인해주세요.")
    print(f"사용 가능한 컬럼: {list(df_park.columns)}")

## 3. 인구 데이터 통합
### 3-1. 서울시 행정동별 인구 데이터 로드

### 3-2. 행정동 경계와 인구 데이터 결합

In [ ]:
# KOSIS 인구 데이터 로드
pop_xlsx = base_path / 'KOSIS' / 'KOSIS_행정동별_성별_연령별_인구.xlsx'
df_pop = pd.read_excel(pop_xlsx)

print(f"인구 데이터 행 수: {len(df_pop)}")
print(f"컬럼: {list(df_pop.columns)}")
print(f"\n상위 5개 데이터:")
df_pop.head()

In [ ]:
# KOSIS 인구 데이터와 gdf_seoul 결합
print(f"KOSIS 데이터 컬럼: {list(df_pop.columns)}")
print(f"gdf_seoul 컬럼: {list(gdf_seoul.columns)}")
print(f"\nKOSIS 데이터 샘플:")
print(df_pop.head(10))
print(f"\nKOSIS 데이터 연령별 고유값:")
print(df_pop['연령별'].unique())

# KOSIS 데이터 전처리: 연령대별로 그룹화
# 연령대 매핑 함수 수정 (더 정확한 매핑)
def map_age_group(age_str):
    if '0~4' in age_str or '5~9' in age_str:
        return '유아청소년'  # 0-9세
    elif '10~14' in age_str or '15~19' in age_str:
        return '유아청소년'  # 10-19세
    elif '20~24' in age_str or '25~29' in age_str:
        return '청년'  # 20-29세
    elif '30~34' in age_str or '35~39' in age_str:
        return '청년'  # 30-39세
    elif '40~44' in age_str or '45~49' in age_str:
        return '중년'  # 40-49세
    elif '50~54' in age_str or '55~59' in age_str:
        return '중년'  # 50-59세
    elif '60~64' in age_str or '65~69' in age_str:
        return '장년'  # 60-69세
    elif '70~74' in age_str or '75~79' in age_str:
        return '장년'  # 70-79세
    elif '80~84' in age_str or '85' in age_str:
        return '장년'  # 80세 이상
    else:
        return '기타'

# 연령대 그룹 추가
df_pop['연령대'] = df_pop['연령별'].apply(map_age_group)
print(f"\n연령대 그룹화 결과:")
print(df_pop[['연령별', '연령대']].drop_duplicates().sort_values('연령별'))

# 행정동별로 연령대별 인구 집계
df_pop_grouped = df_pop.groupby(['행정구역별(읍면동)', '연령대']).agg({
    '총인구(명)': 'sum',
    '총인구_남자(명)': 'sum',
    '총인구_여자(명)': 'sum'
}).reset_index()

print(f"\n그룹화 후 데이터 샘플:")
print(df_pop_grouped.head(20))

# 행정동별 전체 인구도 계산 (모든 연령대 합계)
df_pop_total = df_pop.groupby('행정구역별(읍면동)').agg({
    '총인구(명)': 'sum',
    '총인구_남자(명)': 'sum',
    '총인구_여자(명)': 'sum'
}).reset_index()
df_pop_total['연령대'] = '전체'

# 전체 인구와 연령대별 인구를 합침
df_pop_all = pd.concat([df_pop_grouped, df_pop_total], ignore_index=True)

print(f"\n전체 인구 추가 후:")
print(df_pop_all[df_pop_all['행정구역별(읍면동)'] == '가락1동'])

# 피벗 테이블로 변환: 연령대를 컬럼으로
df_pop_pivot = df_pop_all.pivot(
    index='행정구역별(읍면동)',
    columns='연령대',
    values=['총인구(명)', '총인구_남자(명)', '총인구_여자(명)']
)

# 컬럼명 정리
df_pop_pivot.columns = ['_'.join(col).strip() for col in df_pop_pivot.columns.values]
df_pop_pivot = df_pop_pivot.reset_index()

print(f"\n피벗 후 컬럼:")
print(list(df_pop_pivot.columns))
print(f"\n피벗 후 데이터 샘플:")
print(df_pop_pivot.head())

# gdf_seoul과 결합
gdf_seoul = gdf_seoul.merge(
    df_pop_pivot,
    left_on='ADM_NM',
    right_on='행정구역별(읍면동)',
    how='left'
)

print(f"\n결합 완료!")
print(f"결합 후 데이터 행 수: {len(gdf_seoul)}")
print(f"결합 후 컬럼 수: {len(gdf_seoul.columns)}")
print(f"\n인구 데이터 결측치 확인:")
if '총인구(명)_전체' in gdf_seoul.columns:
    print(f"  총인구(명)_전체 결측: {gdf_seoul['총인구(명)_전체'].isnull().sum()}개")
else:
    print(f"  경고: 총인구(명)_전체 컬럼이 없습니다!")
    print(f"  실제 컬럼: {[col for col in gdf_seoul.columns if '총인구' in col]}")

gdf_seoul.head()

In [ ]:
# 인구 밀도 계산
if '총인구(명)_전체' in gdf_seoul.columns:
    # 총인구 컬럼을 숫자형으로 변환
    gdf_seoul['total_population'] = pd.to_numeric(gdf_seoul['총인구(명)_전체'], errors='coerce')
    gdf_seoul['pop_density_km2'] = gdf_seoul['total_population'] / gdf_seoul['adm_area_km2']
    
    # 연령대별 인구 밀도도 계산
    for age_group in ['유아청소년', '청년', '중년', '장년']:
        col_name = f'총인구(명)_{age_group}'
        if col_name in gdf_seoul.columns:
            gdf_seoul[f'pop_{age_group}'] = pd.to_numeric(gdf_seoul[col_name], errors='coerce')
            gdf_seoul[f'pop_density_{age_group}_km2'] = gdf_seoul[f'pop_{age_group}'] / gdf_seoul['adm_area_km2']
    
    print("=" * 80)
    print("인구 데이터 통계")
    print("=" * 80)
    print(f"\n✓ 사용한 인구 컬럼: 총인구(명)_전체")
    
    print(f"\n【 전체 인구 통계 】")
    print(gdf_seoul['total_population'].describe())
    
    print(f"\n【 전체 인구밀도 통계 (명/km²) 】")
    print(gdf_seoul['pop_density_km2'].describe())
    
    print(f"\n【 연령대별 인구 통계 】")
    for age_group in ['유아청소년', '청년', '중년', '장년']:
        col_name = f'pop_{age_group}'
        if col_name in gdf_seoul.columns:
            print(f"\n  [{age_group}]")
            print(f"    평균: {gdf_seoul[col_name].mean():,.0f}명")
            print(f"    최대: {gdf_seoul[col_name].max():,.0f}명")
            print(f"    최소: {gdf_seoul[col_name].min():,.0f}명")
    
    # 결측치 확인
    null_count = gdf_seoul['total_population'].isnull().sum()
    if null_count > 0:
        print(f"\n⚠ 경고: {null_count}개 행정동에 인구 데이터가 없습니다.")
        print(f"결측 행정동: {gdf_seoul[gdf_seoul['total_population'].isnull()]['ADM_NM'].tolist()}")
    else:
        print(f"\n✓ 모든 행정동에 인구 데이터가 매핑되었습니다!")
    
    print("=" * 80)
else:
    print("경고: 총인구(명)_전체 컬럼을 찾을 수 없습니다.")

# 결과 확인
print(f"\n최종 gdf_seoul 데이터프레임 정보:")
print(f"  - 행 수: {len(gdf_seoul)}")
print(f"  - 컬럼 수: {len(gdf_seoul.columns)}")
print(f"\n주요 컬럼:")
pop_cols = [col for col in gdf_seoul.columns if 'pop' in col.lower() or '인구' in col]
print(f"  {pop_cols}")

gdf_seoul.head()

In [ ]:
# 처리된 데이터 저장
# Shapefile 저장을 위해 컬럼명 영문으로 변경 (Shapefile은 한글 컬럼명 지원 제한적)
gdf_seoul_save = gdf_seoul.copy()

# 컬럼명 매핑 (한글 -> 영문)
column_mapping = {
    '행정구역별(읍면동)': 'dong_name',
    '총인구(명)_전체': 'pop_total',
    '총인구_남자(명)_전체': 'pop_male',
    '총인구_여자(명)_전체': 'pop_female',
    '총인구(명)_유아청소년': 'pop_youth',
    '총인구_남자(명)_유아청소년': 'pop_youth_m',
    '총인구_여자(명)_유아청소년': 'pop_youth_f',
    '총인구(명)_청년': 'pop_young',
    '총인구_남자(명)_청년': 'pop_young_m',
    '총인구_여자(명)_청년': 'pop_young_f',
    '총인구(명)_중년': 'pop_middle',
    '총인구_남자(명)_중년': 'pop_middle_m',
    '총인구_여자(명)_중년': 'pop_middle_f',
    '총인구(명)_장년': 'pop_senior',
    '총인구_남자(명)_장년': 'pop_senior_m',
    '총인구_여자(명)_장년': 'pop_senior_f'
}

# 존재하는 컬럼만 변경
for old_name, new_name in column_mapping.items():
    if old_name in gdf_seoul_save.columns:
        gdf_seoul_save = gdf_seoul_save.rename(columns={old_name: new_name})

print(f"저장용 컬럼명 변경 완료")
print(f"변경된 컬럼: {list(gdf_seoul_save.columns)}")

# 1. 서울시 행정동 경계 + 인구 데이터 저장 (Shapefile)
output_dong = output_path / 'seoul_dong_with_pop.shp'
gdf_seoul_save.to_file(output_dong, encoding='utf-8')
print(f"저장 완료: {output_dong}")

# 2. GeoJSON 형식으로도 저장 (웹 시각화용 - 원본 한글 컬럼명 유지)
output_dong_json = output_path / 'seoul_dong_with_pop.geojson'
gdf_seoul.to_file(output_dong_json, driver='GeoJSON', encoding='utf-8')
print(f"저장 완료: {output_dong_json}")

# 3. 공원 데이터 저장 (폴리곤만 저장)
output_park = output_path / 'seoul_parks.shp'
df_park.to_file(output_park, encoding='utf-8')
print(f"저장 완료: {output_park}")

output_park_json = output_path / 'seoul_parks.geojson'
df_park.to_file(output_park_json, driver='GeoJSON', encoding='utf-8')
print(f"저장 완료: {output_park_json}")

In [ ]:
# 좌표계 확인
print("=" * 80)
print("좌표계 확인")
print("=" * 80)
print(f"\n1. gdf_seoul (행정동 경계) 좌표계:")
print(f"   - CRS: {gdf_seoul.crs}")
print(f"   - EPSG 코드: {gdf_seoul.crs.to_epsg() if gdf_seoul.crs else 'None'}")

print(f"\n2. df_park (공원 데이터) 좌표계:")
print(f"   - CRS: {df_park.crs}")
print(f"   - EPSG 코드: {df_park.crs.to_epsg() if df_park.crs else 'None'}")

# 좌표계가 다른 경우 공원 데이터를 EPSG:5179로 변환
if df_park.crs != 'EPSG:5179':
    print(f"\n공원 데이터 좌표계를 EPSG:5179로 변환 중...")
    df_park = df_park.to_crs('EPSG:5179')
    print(f"변환 완료: {df_park.crs}")
else:
    print(f"\n공원 데이터는 이미 EPSG:5179 좌표계입니다.")

print("\n" + "=" * 80)

### 5-2. gdf_seoul과 공원 데이터 중첩 시각화

In [ ]:
# gdf_seoul 위에 공원 데이터 중첩 시각화
print("=" * 80)
print("행정동 경계와 공원(녹지) 중첩 시각화")
print("=" * 80)

# 시각화 전 데이터 확인
print(f"\n데이터 확인:")
print(f"  - gdf_seoul: {len(gdf_seoul)}개 행정동")
print(f"  - df_park: {len(df_park)}개 공원")
print(f"  - gdf_seoul CRS: {gdf_seoul.crs}")
print(f"  - df_park CRS: {df_park.crs}")

# 두 데이터의 경계 확인
seoul_bounds = gdf_seoul.total_bounds
park_bounds = df_park.total_bounds
print(f"\n경계 확인:")
print(f"  - 서울시 경계: {seoul_bounds}")
print(f"  - 공원 경계: {park_bounds}")

# 시각화
fig, ax = plt.subplots(figsize=(16, 14))

# 1. 서울시 행정동 경계 (배경)
gdf_seoul.plot(
    ax=ax, 
    edgecolor='gray', 
    facecolor='lightgray', 
    linewidth=0.5, 
    alpha=0.6, 
    label='행정동 경계'
)

# 2. 공원 데이터 (녹지정보)
df_park.plot(
    ax=ax, 
    facecolor='green', 
    edgecolor='darkgreen', 
    linewidth=0.3, 
    alpha=0.7, 
    label='공원(녹지)'
)

# 시각화 설정
ax.set_title('서울시 행정동 경계와 공원(녹지) 중첩 시각화', fontsize=20, fontweight='bold', pad=20)
ax.set_xlabel('X (m) - EPSG:5179', fontsize=14)
ax.set_ylabel('Y (m) - EPSG:5179', fontsize=14)
ax.legend(loc='upper right', fontsize=14, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# 축 범위를 서울시 경계에 맞춤
ax.set_xlim(seoul_bounds[0] - 1000, seoul_bounds[2] + 1000)
ax.set_ylim(seoul_bounds[1] - 1000, seoul_bounds[3] + 1000)

plt.tight_layout()
plt.show()

# 데이터 요약 출력
print("\n" + "=" * 80)
print("좌표계 중첩 확인 결과")
print("=" * 80)
print(f"\n✓ gdf_seoul 좌표계: {gdf_seoul.crs}")
print(f"✓ df_park 좌표계: {df_park.crs}")
print(f"\n✓ 두 데이터의 좌표계가 {'동일합니다 (EPSG:5179)' if str(gdf_seoul.crs) == str(df_park.crs) else '다릅니다!'}")
print(f"\n✓ 시각화 확인: 행정동 경계(회색) 위에 공원(녹색)이 정확히 중첩되어 표시됩니다.")
print(f"✓ 총 행정동 수: {len(gdf_seoul)}개")
print(f"✓ 총 공원 수: {len(df_park)}개")
print(f"✓ 총 공원 면적: {df_park['df_park_area_km2'].sum():.2f} km²")
print(f"✓ 서울시 대비 공원 비율: {(df_park['df_park_area_km2'].sum() / gdf_seoul['adm_area_km2'].sum() * 100):.2f}%")
print("=" * 80)

## 6. 데이터 구축 최종 요약

In [ ]:
# 최종 데이터 구축 요약
print("=" * 80)
print("Phase 1: 데이터 구축 및 전처리 완료")
print("=" * 80)

print(f"\n【 1. 서울시 행정동 경계 데이터 】")
print(f"   ├─ 행정동 수: {len(gdf_seoul)}개")
print(f"   ├─ 총 면적: {gdf_seoul['adm_area_km2'].sum():.2f} km²")
print(f"   ├─ 좌표계: {gdf_seoul.crs}")
print(f"   └─ EPSG 코드: {gdf_seoul.crs.to_epsg() if gdf_seoul.crs else 'None'}")

print(f"\n【 2. 공원(녹지) 데이터 】")
print(f"   ├─ 공원 수: {len(df_park)}개")
print(f"   ├─ 총 공원 면적: {df_park['df_park_area_km2'].sum():.2f} km²")
print(f"   ├─ 좌표계: {df_park.crs}")
print(f"   └─ EPSG 코드: {df_park.crs.to_epsg() if df_park.crs else 'None'}")

if 'total_population' in gdf_seoul.columns:
    print(f"\n【 3. 인구 데이터 (KOSIS) 】")
    print(f"   ├─ 데이터 소스: KOSIS_행정동별_성별_연령별_인구.xlsx")
    print(f"   ├─ 총 인구: {gdf_seoul['total_population'].sum():,.0f}명")
    print(f"   ├─ 평균 인구밀도: {gdf_seoul['pop_density_km2'].mean():,.0f} 명/km²")
    print(f"   ├─ 최대 인구밀도: {gdf_seoul['pop_density_km2'].max():,.0f} 명/km²")
    print(f"   └─ 최소 인구밀도: {gdf_seoul['pop_density_km2'].min():,.0f} 명/km²")

print(f"\n【 4. 좌표계 통일 확인 】")
print(f"   ├─ gdf_seoul과 df_park 좌표계 일치: {'✓ YES' if gdf_seoul.crs == df_park.crs else '✗ NO'}")
print(f"   └─ 모두 EPSG:5179로 통일: {'✓ YES' if str(gdf_seoul.crs) == 'EPSG:5179' and str(df_park.crs) == 'EPSG:5179' else '✗ NO'}")

print(f"\n【 5. 저장된 파일 】")
print(f"   ├─ {output_path / 'seoul_dong_with_pop.shp'}")
print(f"   ├─ {output_path / 'seoul_dong_with_pop.geojson'}")
print(f"   ├─ {output_path / 'seoul_parks.shp'}")
print(f"   └─ {output_path / 'seoul_parks.geojson'}")

print("\n" + "=" * 80)
print("✓ 데이터 구축 완료 - Phase 2 (녹지 비율 및 접근성 분석)로 진행 가능")
print("=" * 80)